# Pobieramy dane z https://www.policja.pl/pol/form/1,Informacja-dzienna.html

In [1]:
import requests
import time
import csv
from pathlib import Path
from bs4 import BeautifulSoup
from rich.live import Live

Pobieramy html i trasformujemy do json i zapisujemy na dysk

In [ ]:
PAGES = 300
URL = "https://www.policja.pl/pol/form/1,Informacja-dzienna.html?page={page_number}"
SLEEP_BETWEEN = 1

OUTPUT = Path("../../data/raw/accidents.csv")

COLUMNS = [
    "date",
    "interventions",
    "caught_in_act",
    "wanted_arrested",
    "drunk_drivers",
    "accidents",
    "killed",
    "injured"
]

saved_dates = set()

if OUTPUT.exists():
    with open(OUTPUT, encoding="utf-8", newline="") as f:
        saved_dates = {
            row["date"]
            for row in csv.DictReader(f)
        }

with open(OUTPUT, "a", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=COLUMNS)

    if OUTPUT.stat().st_size == 0:
        writer.writeheader()

    with Live("Starting...") as live:
        for i in range(PAGES):
            url = URL.format(page_number=i)

            try:
                response = requests.get(url, timeout=30)
                response.raise_for_status()

                soup = BeautifulSoup(response.text, "html.parser")

                table = next(
                    (
                        table for table in soup.find_all("table")
                        if "Wypadki drogowe" in table.get_text()
                        and "Interwencje" in table.get_text()
                    ),
                    None
                )

                if table is None:
                    raise ValueError("Statistics table not found")

                count = 0

                for tr in table.find_all("tr"):
                    cells = [
                        td.get_text(" ", strip=True)
                        for td in tr.find_all("td")
                    ]

                    if len(cells) != len(COLUMNS):
                        continue

                    date = cells[0]

                    if not (
                        len(date) == 10
                        and date[4] == "-"
                        and date[7] == "-"
                    ):
                        continue

                    if date in saved_dates:
                        continue

                    row = dict(zip(COLUMNS, cells))

                    for col in COLUMNS[1:]:
                        value = row[col].replace(" ", "").strip()
                        row[col] = int(value) if value else None

                    writer.writerow(row)
                    saved_dates.add(date)
                    count += 1

                f.flush()

                live.update(
                    f"Page {i+1}/{PAGES} | "
                    f"New rows: {count} | "
                    f"Total: {len(saved_dates)}"
                )

            except Exception as e:
                live.update(f"Error on page {i}: {e}")
                time.sleep(30)

            time.sleep(SLEEP_BETWEEN)

Page 30/300 | New rows: 15 | Total: 449